In [35]:
import pandas as pd

from tzlocal import get_localzone
from datetime import datetime
import plotly.graph_objects as go
import warnings
warnings.simplefilter('ignore')
warnings.filterwarnings('ignore')
from tqdm import tqdm
tqdm.pandas()

import train_model as tm
from train_model import ModelFunc
import data_processing as dp
from data_loader import load_data_at_start_date, load_data_by_period
from features import FeatureEngineering

import optuna
from optuna import Trial, visualization
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [36]:
period= -(datetime.now() - datetime(2018, 1, 1)).days
load_data_at_start_date(['BTC-USD'], period, '1d', 'crypto_data')
data = dp.get_data('crypto_data', 'BTC-USD', compress=False)
data = tm.standard_scaler(data)
# display(data)

Start load data, tickers ['BTC-USD'], interval: 1d, start date: -2604
date: 2018-01-01 08:00:10.438597


[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2604 entries, 2018-01-01 to 2025-02-17
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   (BTC-USD, Open)       2604 non-null   float64
 1   (BTC-USD, High)       2604 non-null   float64
 2   (BTC-USD, Low)        2604 non-null   float64
 3   (BTC-USD, Close)      2604 non-null   float64
 4   (BTC-USD, Adj Close)  2604 non-null   float64
 5   (BTC-USD, Volume)     2604 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 142.4 KB
Download data completed


In [37]:
fe_params = {
    'emaf': 20,
    'emam': 100,
    'emas': 150,
    'rsi': 14,
    'macd': [12, 26, 9],
 }

fe = FeatureEngineering(fe_params)

In [38]:
lag_periods = 3
features_to_trend = ['Open', 'High', 'Low', 'Close', 'Volume']
data = fe.clear_invalid_targets(fe.add_target(fe.enrich_with_indicators(data), lag_periods))
data_with_trend, new_trend_features = fe.create_trend_features(data, features_to_trend, lag_periods) 
data = data_with_trend[new_trend_features + ['Target']]


In [39]:
split_param = {
    'last_test_month_cnt': 1,
    'last_val_month_cnt': 2,
}

train_data, val_data, test_data = tm.split_data_by_date2(data, split_param)

X_train, y_train = tm.split_by_features_and_target_variables(train_data, new_trend_features)
X_val, y_val = tm.split_by_features_and_target_variables(val_data, new_trend_features)
X_test, y_test = tm.split_by_features_and_target_variables(test_data, new_trend_features)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

Train size: 1974, Val size: 24, Test size: 27


In [40]:
model = tm.fit_models([ModelFunc.CATBOOST_CLASS], X_train, y_train)[0]

models_proba = tm.predict_models([model], X_train, X_val, X_test)[0]
y_train_pred_prob = models_proba['train']
y_val_pred_prob = models_proba['val']
y_test_pred_prob = models_proba['test']

train_roc_auc = tm.roc_auc_score_metric(y_train, y_train_pred_prob)
val_roc_auc = tm.roc_auc_score_metric(y_val, y_val_pred_prob)
test_roc_auc = tm.roc_auc_score_metric(y_test, y_test_pred_prob)

In [41]:
print("=== Метрики для обучающей выборки ===")
print(f"ROC AUC: {train_roc_auc:.4f}")
tm.calculate_metrics_table(y_train, y_train_pred_prob)


=== Метрики для обучающей выборки ===
ROC AUC: 1.0000


,Cutoff,Precision,Recall,Accuracy,F1-Score
0,50.0,100.0,100.0,100.0,100.0
1,60.0,100.0,100.0,100.0,100.0
2,70.0,100.0,100.0,100.0,100.0
3,80.0,100.0,100.0,100.0,100.0


In [42]:
print("=== Метрики для валидационной выборки ===")
print(f"ROC AUC: {val_roc_auc:.4f}")
tm.calculate_metrics_table(y_val, y_val_pred_prob)

=== Метрики для валидационной выборки ===
ROC AUC: 0.5500


,Cutoff,Precision,Recall,Accuracy,F1-Score
0,50.0,46.153846,60.0,54.166667,52.173913
1,60.0,45.454545,50.0,54.166667,47.619048
2,70.0,42.857143,30.0,54.166667,35.294118
3,80.0,40.000000,20.0,54.166667,26.666667


In [43]:
print("=== Метрики для тестовой выборки ===")
print(f"ROC AUC: {test_roc_auc:.4f}")
tm.calculate_metrics_table(y_test, y_test_pred_prob)

=== Метрики для тестовой выборки ===
ROC AUC: 0.4545


,Cutoff,Precision,Recall,Accuracy,F1-Score
0,50.0,38.888889,63.636364,44.444444,48.275862
1,60.0,37.500000,54.545455,44.444444,44.444444
2,70.0,30.769231,36.363636,40.740741,33.333333
3,80.0,40.000000,36.363636,51.851852,38.095238


### Hyperparameter tuning with GridSearchCV

In [44]:
params = {
    'grid_params': {
        'depth': [4, 6, 8, 10],                      # Разные значения глубины дерева
        'learning_rate': [0.01, 0.05, 0.1, 0.2],     # Разные темпы обучения
        'n_estimators': [100, 200, 500],             # Разные количества деревьев
        'l2_leaf_reg': [1, 3, 5, 7],                 # Разные коэффициенты L2-регуляризации
        #takes long time to run
        # 'bagging_temperature': [0, 0.3, 0.6, 1],   # Разные температуры бэггинга
        # 'rsm': [0.5, 0.8, 1.0],                    # Разные доли признаков (RSM)
        # 'subsample': [0.5, 0.8, 1.0],              # Разные доли выборки для каждого дерева
    },
    'params': {
        'scoring': 'roc_auc', # accuracy
        'cv': 5,                        # Количество фолдов для кросс-валидации
        'n_jobs': -1,
        'random_state': 42,
        'verbose': 0,
        'early_stopping_rounds': 50, # Activates Iter overfitting detector with od_wait parameter 
        'task_type': 'GPU',
    }
}

grid_search = tm.catboot_classifier_model_grid_search(X_train, y_train, X_val, y_val, params)

# Получаем лучшую модель и параметры
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("Best params:", best_params)
print("Best ROC AUC Score:", best_score)


Best params: {'depth': 10, 'l2_leaf_reg': 3, 'learning_rate': 0.1, 'n_estimators': 100}
Best ROC AUC Score: 0.5018210904556477


### Optuna: hyperparameter optimization

In [45]:
study = tm.optuna_study_CatBoostClassifier(X_train, y_train, X_val, y_val, n_trials=5)

best_params = study.best_params
best_score = study.best_value

print("best params:", best_params)
print("best ROC AUC Score:", best_score)

best params: {'iterations': 153, 'depth': 6, 'learning_rate': 0.04579985421561353, 'l2_leaf_reg': 0.023287253835155276, 'bagging_temperature': 0.16892818709022095, 'rsm': 0.832781645234173, 'subsample': 0.6092371476590615}
best ROC AUC Score: 0.6495461038072738


In [46]:
#shows the relative importances of hyperparameters.
tm.optuna_plot_param_importances(study)


In [47]:
#plots the empirical distribution function of the objective.
tm.optuna_plot_edf(study)

### Using best params

In [48]:
tm.model_fit_with_eval(ModelFunc.CATBOOST_CLASS, X_train, y_train, (X_val, y_val), best_params)

models_proba = tm.predict_models([model], X_train, X_val, X_test)[0]
y_train_pred_prob = models_proba['train']
y_val_pred_prob = models_proba['val']
y_test_pred_prob = models_proba['test']

train_roc_auc = tm.roc_auc_score_metric(y_train, y_train_pred_prob)
val_roc_auc = tm.roc_auc_score_metric(y_val, y_val_pred_prob)
test_roc_auc = tm.roc_auc_score_metric(y_test, y_test_pred_prob)

In [49]:
print("=== Train sample metrics ===")
print(f"ROC AUC: {train_roc_auc:.4f}")
tm.calculate_metrics_table(y_train, y_train_pred_prob)

=== Train sample metrics ===
ROC AUC: 1.0000


,Cutoff,Precision,Recall,Accuracy,F1-Score
0,50.0,100.0,100.0,100.0,100.0
1,60.0,100.0,100.0,100.0,100.0
2,70.0,100.0,100.0,100.0,100.0
3,80.0,100.0,100.0,100.0,100.0


In [50]:
print("=== Validation sample metrics ===")
print(f"ROC AUC: {val_roc_auc:.4f}")
tm.calculate_metrics_table(y_val, y_val_pred_prob)

=== Validation sample metrics ===
ROC AUC: 0.5500


,Cutoff,Precision,Recall,Accuracy,F1-Score
0,50.0,46.153846,60.0,54.166667,52.173913
1,60.0,45.454545,50.0,54.166667,47.619048
2,70.0,42.857143,30.0,54.166667,35.294118
3,80.0,40.000000,20.0,54.166667,26.666667


In [51]:
print("=== Test sample metrics ===")
print(f"ROC AUC: {test_roc_auc:.4f}")
tm.calculate_metrics_table(y_test, y_test_pred_prob)

=== Test sample metrics ===
ROC AUC: 0.4545


,Cutoff,Precision,Recall,Accuracy,F1-Score
0,50.0,38.888889,63.636364,44.444444,48.275862
1,60.0,37.500000,54.545455,44.444444,44.444444
2,70.0,30.769231,36.363636,40.740741,33.333333
3,80.0,40.000000,36.363636,51.851852,38.095238
